In [ ]:
#%pip install -q ultralytics opencv-python-headless scipy tqdm

import os, math, json, random, collections, re, datetime
import numpy as np
import cv2
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from pathlib import Path
from tqdm import tqdm


In [ ]:
# ====== CONFIG ======
# ----- I/O -----
INPUT_VIDEO   = "/content/drive/MyDrive/pig_data_unzipped/pigs011219/PIGS011219/000430/color.mp4"
OUTPUT_VIDEO  = "/content/drive/MyDrive/output_tracked.mp4"       # video ra
WEIGHTS       = "/content/weights.pt"               # YOLOv8 .pt
MASK_PATH     = "/content/drive/MyDrive/pig_data_unzipped/mask.png"
TIME_FILE     = "/content/drive/MyDrive/pig_data_unzipped/pigs011219/PIGS011219/000430/times.txt"

CONF_THRESH   = 0.25
NMS_IOU       = 0.70
ALLOWED_CLASS = None

# ROI
USE_ROI       = True
ROI_MODE      = "center"   # "center" | "cover"
ROI_MIN_COVER = 0.10
ROI_DILATE_PX = 8

# Appearance bank (per-ID)
H_BINS, S_BINS, V_BINS = 16, 16, 4
APP_W, APP_H           = 96, 96
ID_BANK_SIZE           = 60

ID_POOL        = list(range(1, 9))   # 1..8
REID_THR       = 0.20
ID_STEAL_FIX   = True

DRAW_THICKNESS = 2
FONT_SCALE     = 0.7

BYTE_TRACK_YAML = "/content/bytetrack_custom.yaml"

In [ ]:
%%bash
cat > /content/bytetrack_custom.yaml <<'YAML'
tracker_type: bytetrack
track_high_thresh: 0.6
track_low_thresh: 0.1
new_track_thresh: 0.6
match_thresh: 0.8
track_buffer: 60
min_box_area: 10
mot20: false
YAML


In [ ]:
def load_mask(path, W, H):
    if not path or not os.path.exists(path):
        return None
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    if ROI_DILATE_PX > 0:
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ROI_DILATE_PX, ROI_DILATE_PX))
        m = cv2.dilate(m, ker, 1)
    return cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)

def apply_visual_mask(frame, mask):
    if mask is None: return frame
    if len(mask.shape)==2:
        m3 = cv2.merge([mask, mask, mask])
    else:
        m3 = mask
    out = cv2.bitwise_and(frame, m3)
    return out

def roi_keep(mask, box):
    if mask is None: return True
    x1,y1,x2,y2 = map(int, box)
    H, W = mask.shape[:2]
    x1=max(0,min(W-1,x1)); x2=max(0,min(W-1,x2))
    y1=max(0,min(H-1,y1)); y2=max(0,min(H-1,y2))
    if x2<=x1 or y2<=y1: return False
    if ROI_MODE=="center":
        cx=(x1+x2)//2; cy=(y1+y2)//2
        return mask[cy, cx]==255
    # cover
    roi = mask[y1:y2, x1:x2]
    if roi.size==0: return False
    cover = np.count_nonzero(roi==255)/float(roi.size)
    return cover >= ROI_MIN_COVER

def extract_hist_hsv(frame, box):
    x1,y1,x2,y2 = map(int, box)
    H, W = frame.shape[:2]
    x1=max(0,min(W-1,x1)); x2=max(0,min(W-1,x2))
    y1=max(0,min(H-1,y1)); y2=max(0,min(H-1,y2))
    if x2<=x1 or y2<=y1:
        return np.ones((H_BINS*S_BINS*V_BINS,), np.float32)/(H_BINS*S_BINS*V_BINS)
    crop = frame[y1:y2, x1:x2]
    if crop.size==0:
        return np.ones((H_BINS*S_BINS*V_BINS,), np.float32)/(H_BINS*S_BINS*V_BINS)
    crop = cv2.resize(crop, (APP_W, APP_H))
    hsv  = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv],[0,1,2],None,[H_BINS,S_BINS,V_BINS],[0,180,0,256,0,256]).astype(np.float32)
    hist /= (hist.sum() + 1e-6)
    return hist.flatten()

def bhattacharyya(h1, h2):
    return np.clip(1.0 - np.sum(np.sqrt(h1*h2)), 0.0, 1.0)

class IDBank:
    def __init__(self, id_pool, bank_size=60):
        self.bank_size = bank_size
        self.id_pool = sorted(id_pool)
        self.bank = {fid: collections.deque(maxlen=bank_size) for fid in self.id_pool}
    def update(self, fid, hist):
        if fid is None: return
        self.bank.setdefault(fid, collections.deque(maxlen=self.bank_size)).append(hist)
    def distance_to_id(self, hist, fid):
        arr = self.bank.get(fid, None)
        if not arr or len(arr)==0: return 1.0
        mean_hist = np.mean(np.stack(arr,0), axis=0)
        return bhattacharyya(hist, mean_hist)
    def best_id(self, hist):
        best_f, best_d = None, 1.0
        for fid in self.id_pool:
            d = self.distance_to_id(hist, fid)
            if d < best_d:
                best_d, best_f = d, fid
        return best_f, best_d

id_bank = IDBank(ID_POOL, ID_BANK_SIZE)

# Pool & mapping (raw -> fixed 1..8)
free_ids = set(ID_POOL)
raw_to_fixed = {}   # raw track id (ByteTrack) -> fixed id
fixed_to_raw = {}   # fixed id -> raw track id

def take_free_id():
    if free_ids:
        fid = min(free_ids)
        free_ids.remove(fid)
        return fid
    return None

def bind_fixed(fid, rid):
    if fid in fixed_to_raw and fixed_to_raw[fid] != rid:
        old = fixed_to_raw[fid]
        if old in raw_to_fixed: del raw_to_fixed[old]
    fixed_to_raw[fid] = rid
    raw_to_fixed[rid] = fid

def free_fixed_by_raw(rid):
    if rid in raw_to_fixed:
        fid = raw_to_fixed[rid]
        free_ids.add(fid)
        del raw_to_fixed[rid]
        if fid in fixed_to_raw: del fixed_to_raw[fid]

def reassign_ids_consistency(active_detections):
    """
    active_detections: list of dict {rid, hist}
    """
    if not active_detections: return
    rids = [d["rid"] for d in active_detections]
    n, m = len(rids), len(ID_POOL)
    C = np.zeros((n,m), np.float32)
    for i, det in enumerate(active_detections):
        hist = det["hist"]
        for j, fid in enumerate(ID_POOL):
            d = id_bank.distance_to_id(hist, fid)
            pen = 0.15 if (fid in fixed_to_raw and fixed_to_raw[fid]!=det["rid"]) else 0.0
            C[i,j] = min(1.0, d + pen)
    r,c = linear_sum_assignment(C)
    taken = set()
    for i,j in zip(r,c):
        rid = rids[i]; fid = ID_POOL[j]
        if fid in taken: continue
        cur = raw_to_fixed.get(rid, None)
        if cur == fid:
            taken.add(fid); continue
        if cur is not None:
            if cur in fixed_to_raw: del fixed_to_raw[cur]
        if fid in fixed_to_raw:
            other = fixed_to_raw[fid]
            if other in raw_to_fixed: del raw_to_fixed[other]
        bind_fixed(fid, rid)
        if fid in free_ids: free_ids.remove(fid)
        taken.add(fid)


In [ ]:
%%bash
cat > /content/bytetrack_custom.yaml <<'YAML'
tracker_type: bytetrack
track_thresh: 0.6                 # ~= track_high_thresh
new_track_thresh: 0.6
track_buffer: 60
match_thresh: 0.8
min_box_area: 10
mot20: false

fuse_score: true
proximity_thresh: 0.5
appearance_thresh: 0.25
max_age: 60
n_init: 3
with_reid: false
YAML


In [ ]:
def load_mask(path, W, H):
    if not path or not os.path.exists(path):
        return None
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    if ROI_DILATE_PX > 0:
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ROI_DILATE_PX, ROI_DILATE_PX))
        m = cv2.dilate(m, ker, 1)
    return cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)

def apply_visual_mask(frame, mask):
    if mask is None: return frame
    if len(mask.shape)==2:
        m3 = cv2.merge([mask, mask, mask])
    else:
        m3 = mask
    out = cv2.bitwise_and(frame, m3)
    return out

def roi_keep(mask, box):
    if mask is None: return True
    x1,y1,x2,y2 = map(int, box)
    H, W = mask.shape[:2]
    x1=max(0,min(W-1,x1)); x2=max(0,min(W-1,x2))
    y1=max(0,min(H-1,y1)); y2=max(0,min(H-1,y2))
    if x2<=x1 or y2<=y1: return False
    if ROI_MODE=="center":
        cx=(x1+x2)//2; cy=(y1+y2)//2
        return mask[cy, cx]==255
    # cover
    roi = mask[y1:y2, x1:x2]
    if roi.size==0: return False
    cover = np.count_nonzero(roi==255)/float(roi.size)
    return cover >= ROI_MIN_COVER

def extract_hist_hsv(frame, box):
    x1,y1,x2,y2 = map(int, box)
    H, W = frame.shape[:2]
    x1=max(0,min(W-1,x1)); x2=max(0,min(W-1,x2))
    y1=max(0,min(H-1,y1)); y2=max(0,min(H-1,y2))
    if x2<=x1 or y2<=y1:
        return np.ones((H_BINS*S_BINS*V_BINS,), np.float32)/(H_BINS*S_BINS*V_BINS)
    crop = frame[y1:y2, x1:x2]
    if crop.size==0:
        return np.ones((H_BINS*S_BINS*V_BINS,), np.float32)/(H_BINS*S_BINS*V_BINS)
    crop = cv2.resize(crop, (APP_W, APP_H))
    hsv  = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv],[0,1,2],None,[H_BINS,S_BINS,V_BINS],[0,180,0,256,0,256]).astype(np.float32)
    hist /= (hist.sum() + 1e-6)
    return hist.flatten()

def bhattacharyya(h1, h2):
    return np.clip(1.0 - np.sum(np.sqrt(h1*h2)), 0.0, 1.0)

class IDBank:
    def __init__(self, id_pool, bank_size=60):
        self.bank_size = bank_size
        self.id_pool = sorted(id_pool)
        self.bank = {fid: collections.deque(maxlen=bank_size) for fid in self.id_pool}
    def update(self, fid, hist):
        if fid is None: return
        self.bank.setdefault(fid, collections.deque(maxlen=self.bank_size)).append(hist)
    def distance_to_id(self, hist, fid):
        arr = self.bank.get(fid, None)
        if not arr or len(arr)==0: return 1.0
        mean_hist = np.mean(np.stack(arr,0), axis=0)
        return bhattacharyya(hist, mean_hist)
    def best_id(self, hist):
        best_f, best_d = None, 1.0
        for fid in self.id_pool:
            d = self.distance_to_id(hist, fid)
            if d < best_d:
                best_d, best_f = d, fid
        return best_f, best_d

id_bank = IDBank(ID_POOL, ID_BANK_SIZE)

# Pool & mapping (raw -> fixed 1..8)
free_ids = set(ID_POOL)
raw_to_fixed = {}   # raw track id (ByteTrack) -> fixed id
fixed_to_raw = {}   # fixed id -> raw track id

def take_free_id():
    if free_ids:
        fid = min(free_ids)
        free_ids.remove(fid)
        return fid
    return None

def bind_fixed(fid, rid):
    if fid in fixed_to_raw and fixed_to_raw[fid] != rid:
        old = fixed_to_raw[fid]
        if old in raw_to_fixed: del raw_to_fixed[old]
    fixed_to_raw[fid] = rid
    raw_to_fixed[rid] = fid

def free_fixed_by_raw(rid):
    if rid in raw_to_fixed:
        fid = raw_to_fixed[rid]
        free_ids.add(fid)
        del raw_to_fixed[rid]
        if fid in fixed_to_raw: del fixed_to_raw[fid]

def reassign_ids_consistency(active_detections):
    """
    active_detections: list of dict {rid, hist}
    """
    if not active_detections: return
    rids = [d["rid"] for d in active_detections]
    n, m = len(rids), len(ID_POOL)
    C = np.zeros((n,m), np.float32)
    for i, det in enumerate(active_detections):
        hist = det["hist"]
        for j, fid in enumerate(ID_POOL):
            d = id_bank.distance_to_id(hist, fid)
            pen = 0.15 if (fid in fixed_to_raw and fixed_to_raw[fid]!=det["rid"]) else 0.0
            C[i,j] = min(1.0, d + pen)
    r,c = linear_sum_assignment(C)
    taken = set()
    for i,j in zip(r,c):
        rid = rids[i]; fid = ID_POOL[j]
        if fid in taken: continue
        cur = raw_to_fixed.get(rid, None)
        if cur == fid:
            taken.add(fid); continue
        if cur is not None:
            if cur in fixed_to_raw: del fixed_to_raw[cur]
        if fid in fixed_to_raw:
            other = fixed_to_raw[fid]
            if other in raw_to_fixed: del raw_to_fixed[other]
        bind_fixed(fid, rid)
        if fid in free_ids: free_ids.remove(fid)
        taken.add(fid)


In [ ]:
%%bash
cat > /content/bytetrack_custom.yaml <<'YAML'
# === ByteTrack config compatible with Ultralytics 8.3.225 ===
tracker_type: bytetrack

# -- Newer keys expected by some builds --
track_high_thresh: 0.6
track_low_thresh:  0.1
new_track_thresh:  0.6

track_thresh:      0.6
match_thresh:      0.8
track_buffer:      60
min_box_area:      10
mot20:             false

fuse_score:        true
proximity_thresh:  0.5
appearance_thresh: 0.25
max_age:           60
n_init:            3
with_reid:         false
YAML


In [ ]:
import ultralytics
from ultralytics import YOLO
import cv2, numpy as np
from tqdm import tqdm
import datetime, re, os

print("Ultralytics version:", ultralytics.__version__)

def run_stream(tracker_yaml):
    results = model.track(
        source=INPUT_VIDEO,
        tracker=tracker_yaml,
        conf=CONF_THRESH,
        iou=NMS_IOU,
        stream=True,
        persist=True,
        verbose=False
    )
    return results

cap0 = cv2.VideoCapture(INPUT_VIDEO)
assert cap0.isOpened(), f"Cannot open: {INPUT_VIDEO}"
fps  = cap0.get(cv2.CAP_PROP_FPS) or 30.0
W    = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
H    = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap0.release()

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (W,H))

mask = load_mask(MASK_PATH, W, H) if USE_ROI else None
model = YOLO(WEIGHTS)

time_lines = None; time_header=None
if os.path.exists(TIME_FILE):
    with open(TIME_FILE, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    if lines:
        time_lines = lines
        head = " ".join(lines)
        m = re.search(r"start\s*=\s*([0-9:\-\s\.TZ\+]+).*fps\s*=\s*([0-9\.]+)", head, re.IGNORECASE)
        if m:
            try:
                start = m.group(1).strip()
                fps_s = float(m.group(2))
                if "T" in start:
                    base = datetime.datetime.fromisoformat(start.replace("Z","+00:00"))
                else:
                    base = datetime.datetime.fromisoformat(start)
                time_header = ("header", base, fps_s)
            except:
                time_header = None

def format_ts(idx):
    if time_lines is None: return f"frame {idx}"
    if time_header is None:
        if len(time_lines) >= idx:
            return time_lines[idx-1]
        return f"frame {idx}"
    # header mode
    _, base, fps_s = time_header
    delta = datetime.timedelta(seconds=(idx-1)/fps_s)
    return (base + delta).isoformat(sep=" ", timespec="milliseconds")

id_bank = IDBank(ID_POOL, ID_BANK_SIZE)
free_ids = set(ID_POOL)
raw_to_fixed, fixed_to_raw = {}, {}

def take_free_id():
    if free_ids:
        fid = min(free_ids)
        free_ids.remove(fid)
        return fid
    return None

def bind_fixed(fid, rid):
    if fid in fixed_to_raw and fixed_to_raw[fid] != rid:
        old = fixed_to_raw[fid]
        if old in raw_to_fixed: del raw_to_fixed[old]
    fixed_to_raw[fid] = rid
    raw_to_fixed[rid] = fid

def reassign_ids_consistency(active_detections):
    if not active_detections: return
    rids = [d["rid"] for d in active_detections]
    C = np.zeros((len(rids), len(ID_POOL)), np.float32)
    for i, det in enumerate(active_detections):
        hist = det["hist"]
        for j, fid in enumerate(ID_POOL):
            d = id_bank.distance_to_id(hist, fid)
            pen = 0.15 if (fid in fixed_to_raw and fixed_to_raw[fid]!=det["rid"]) else 0.0
            C[i,j] = min(1.0, d + pen)
    r,c = linear_sum_assignment(C)
    taken = set()
    for i,j in zip(r,c):
        rid = rids[i]; fid = ID_POOL[j]
        if fid in taken: continue
        cur = raw_to_fixed.get(rid, None)
        if cur == fid:
            taken.add(fid); continue
        if cur is not None and cur in fixed_to_raw:
            del fixed_to_raw[cur]
        if fid in fixed_to_raw:
            other = fixed_to_raw[fid]
            if other in raw_to_fixed: del raw_to_fixed[other]
        bind_fixed(fid, rid)
        if fid in free_ids: free_ids.discard(fid)
        taken.add(fid)

try:
    results = run_stream(BYTE_TRACK_YAML)
except AttributeError as e:
    pass
    results = model.track(
        source=INPUT_VIDEO,
        tracker='bytetrack.yaml',
        conf=CONF_THRESH,
        iou=NMS_IOU,
        stream=True,
        persist=True,
        verbose=False
    )

frame_idx = 0
pbar = tqdm(desc="ByteTrack stream", total=None)

for res in results:
    frame_idx += 1
    frame = res.orig_img.copy()
    Hc, Wc = frame.shape[:2]
    if (Hc, Wc) != (H, W):
        H, W = Hc, Wc
        writer.release()
        writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (W,H))
        mask = load_mask(MASK_PATH, W, H) if USE_ROI else None

    dets = []
    if res.boxes is not None and len(res.boxes)>0:
        xyxys = res.boxes.xyxy.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        ids    = res.boxes.id
        ids    = ids.cpu().numpy().astype(int) if ids is not None else None
        clss   = res.boxes.cls.cpu().numpy() if res.boxes.cls is not None else None
        for i, box in enumerate(xyxys):
            if ids is None: continue
            rid = int(ids[i]); score=float(confs[i])
            if (ALLOWED_CLASS is not None) and (int(clss[i])!=int(ALLOWED_CLASS)):
                continue
            if USE_ROI and not roi_keep(mask, box):
                continue
            dets.append({"rid": rid, "box": box, "score": score})

    vis = apply_visual_mask(frame, mask) if USE_ROI else frame

    active_for_opt = []
    for d in dets:
        d["hist"] = extract_hist_hsv(frame, d["box"])
        if d["rid"] not in raw_to_fixed:
            best_id, dist = id_bank.best_id(d["hist"])
            if (best_id is not None) and (dist <= REID_THR) and (best_id in free_ids):
                bind_fixed(best_id, d["rid"])
                if best_id in free_ids: free_ids.discard(best_id)
            else:
                fid = take_free_id()
                if fid is not None: bind_fixed(fid, d["rid"])
        active_for_opt.append({"rid": d["rid"], "hist": d["hist"]})

    if ID_STEAL_FIX:
        reassign_ids_consistency(active_for_opt)

    for d in dets:
        fid = raw_to_fixed.get(d["rid"], None)
        if fid is not None:
            id_bank.update(fid, d["hist"])

    ts = format_ts(frame_idx)
    cv2.putText(vis, ts, (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2, cv2.LINE_AA)
    for d in dets:
        x1,y1,x2,y2 = map(int, d["box"])
        fid = raw_to_fixed.get(d["rid"], None)
        color = (37*(fid or 1)%255, 97*(fid or 1)%255, 173*(fid or 1)%255)
        cv2.rectangle(vis, (x1,y1), (x2,y2), color, DRAW_THICKNESS)
        cv2.putText(vis, f"ID {fid if fid is not None else d['rid']}",
                    (x1, max(0,y1-6)), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, color, 2, cv2.LINE_AA)

    writer.write(vis)
    pbar.update(1)

pbar.close()
writer.release()
print(f"[OK] Wrote video -> {OUTPUT_VIDEO}")
